In [2]:
import os
!pip install -q transformers peft accelerate sentencepiece "torchao>=0.16.0"
if not os.path.exists('/content/llama.cpp'):
    !git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q /content/llama.cpp/gguf-py


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_DIR = "/content/drive/MyDrive/garsonbot_runs/wbot_v3/adapter"
MERGED_DIR  = "/content/wbot_v3_merged"
GGUF_PATH   = "/content/drive/MyDrive/garsonbot_runs/wbot_v3/Qwen3-4B-wbot_v3-Q4_K_M.gguf"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Base model yükleniyor (GPU)...")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    dtype=torch.float16,
    device_map="cuda:0"
)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
print("Merge ediliyor...")
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR)
AutoTokenizer.from_pretrained(ADAPTER_DIR).save_pretrained(MERGED_DIR)
print("Merge tamam:", MERGED_DIR)


Base model yükleniyor (GPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Merge ediliyor...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merge tamam: /content/wbot_v3_merged


In [1]:
import torch, gc
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU serbest: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

GPU serbest: 14.5 GB


In [5]:
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outtype q4_k_m \
    --outfile {GGUF_PATH}
print("GGUF kaydedildi:", GGUF_PATH)

usage: convert_hf_to_gguf.py [-h] [--vocab-only] [--outfile OUTFILE]
                             [--outtype {f32,f16,bf16,q8_0,tq1_0,tq2_0,auto}]
                             [--bigendian] [--use-temp-file] [--no-lazy]
                             [--model-name MODEL_NAME] [--verbose]
                             [--split-max-tensors SPLIT_MAX_TENSORS]
                             [--split-max-size SPLIT_MAX_SIZE] [--dry-run]
                             [--no-tensor-first-split] [--metadata METADATA]
                             [--print-supported-models] [--remote] [--mmproj]
                             [--mtp] [--no-mtp] [--mistral-format]
                             [--disable-mistral-community-chat-template]
                             [--sentence-transformers-dense-modules]
                             [--fuse-gate-up-exps] [--fp8-as-q8]
                             [model]
convert_hf_to_gguf.py: error: argument --outtype: invalid choice: 'q4_k_m' (choose from f32, f16, bf16,

In [6]:
F16_PATH = "/content/wbot_v3_f16.gguf"
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outtype f16 \
    --outfile {F16_PATH}
print("f16 GGUF hazır:", F16_PATH)

INFO:hf-to-gguf:Loading model: wbot_v3_merged
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {2560, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {2560}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {9728, 2560}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {2560, 9728}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {2560, 9728}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {2560}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight,  torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {2560, 1024}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float16

In [7]:
!cmake -B /content/llama.cpp/build -DGGML_CUDA=OFF /content/llama.cpp -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF
!cmake --build /content/llama.cpp/build --target llama-quantize -j4
print("llama-quantize hazır")


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "

In [8]:
!/content/llama.cpp/build/bin/llama-quantize {F16_PATH} {GGUF_PATH} Q4_K_M
print("GGUF kaydedildi:", GGUF_PATH)


llama_print_build_info: build = 9518 (7c158fbb4)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/wbot_v3_f16.gguf' to '/content/drive/MyDrive/garsonbot_runs/wbot_v3/Qwen3-4B-wbot_v3-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 28 key-value pairs and 398 tensors from /content/wbot_v3_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.600000
llama_model_loade